# Umubyeyi: Kinyarwanda maternal-wellness intent classification

Complete pipeline: a baseline and three models, with comparison and analysis. Set the runtime to GPU (Runtime > Change runtime type > T4 GPU), then Run all. All metrics and figures are saved under reports/ and bundled into a zip you can download into the repo at the end.

## Setup

In [ ]:
!pip install -q -U transformers datasets accelerate sentence-transformers scikit-learn

In [ ]:
import warnings, os, json
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

RNG = 42
np.random.seed(RNG); torch.manual_seed(RNG)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE if DEVICE == "cuda" else "CPU (set the runtime to T4 GPU for Model 3)")

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

REPORTS = "reports"
FIGURES = os.path.join(REPORTS, "figures")
os.makedirs(FIGURES, exist_ok=True)

def save_json(filename, payload):
    path = os.path.join(REPORTS, filename)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    print("saved", path)

def save_fig(name):
    path = os.path.join(FIGURES, name)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print("saved", path)

URL = "https://raw.githubusercontent.com/IrutingaboRaissa/UMUBYEYI/main/notebooks/amod_kinyarwanda.csv"
df = pd.read_csv(URL).dropna(subset=["Context", "context_rw", "intent"])
labels = sorted(df["intent"].unique())
l2i = {l: i for i, l in enumerate(labels)}
y = df["intent"].values
yi = df["intent"].map(l2i).values
EN = df["Context"].values
RW = df["context_rw"].values
idx = np.arange(len(df))
tr, te = train_test_split(idx, test_size=0.20, random_state=RNG, stratify=y)
print("rows:", len(df), "| classes:", len(labels), "| train:", len(tr), "| test:", len(te))

results = {}
def score(yt, yp):
    pr, rc, f1, _ = precision_recall_fscore_support(yt, yp, average="macro", zero_division=0)
    return {"accuracy": round(float(accuracy_score(yt, yp)), 4), "precision_macro": round(float(pr), 4),
            "recall_macro": round(float(rc), 4), "f1_macro": round(float(f1), 4)}

## Data overview

In [ ]:
counts = df["intent"].value_counts()
print(counts.to_string())
ax = counts.plot.bar(figsize=(8, 4), color="#2A9D8F", edgecolor="white", rot=20)
ax.set_ylabel("examples"); ax.set_title("Intent class balance")
plt.tight_layout(); save_fig("class_balance.png"); plt.show()

words_en = df["Context"].str.split().str.len()
words_rw = df["context_rw"].str.split().str.len()
print(f"median words  English={int(words_en.median())}  Kinyarwanda={int(words_rw.median())}")
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist([words_en, words_rw], bins=20, label=["English", "Kinyarwanda"], color=["#2A9D8F", "#E76F51"])
ax.set_xlabel("words per question"); ax.set_ylabel("count"); ax.legend(); ax.set_title("Question length")
plt.tight_layout(); save_fig("text_length.png"); plt.show()

## Distinctive terms per intent

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer as _TV
_v = _TV(stop_words="english", min_df=3, max_features=3000)
_M = _v.fit_transform(df["Context"])
_terms = np.array(_v.get_feature_names_out())
for intent in labels:
    mask = (df["intent"].values == intent)
    mean = np.asarray(_M[mask].mean(axis=0)).ravel()
    print(f"{intent:22s} " + ", ".join(_terms[mean.argsort()[::-1][:8]]))

## Baseline: Naive Bayes (English TF-IDF)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

tfidf_en = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2, max_features=8000, sublinear_tf=True)
Xtr = tfidf_en.fit_transform(EN[tr])
Xte = tfidf_en.transform(EN[te])
nb = MultinomialNB().fit(Xtr, y[tr])
results["Baseline (Naive Bayes)"] = score(y[te], nb.predict(Xte))
print("Baseline (Naive Bayes):", results["Baseline (Naive Bayes)"])
save_json("naive_bayes_baseline.json", {"Naive Bayes (baseline)": results["Baseline (Naive Bayes)"]})

## Model 1: TF-IDF + Logistic Regression (English)

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RNG).fit(Xtr, y[tr])
p1 = clf.predict(Xte)
results["Model 1: TF-IDF + LogReg (English)"] = score(y[te], p1)
print("Model 1:", results["Model 1: TF-IDF + LogReg (English)"], "\n")
print(classification_report(y[te], p1, zero_division=0))
save_json("baseline_metrics.json", {"Logistic Regression": results["Model 1: TF-IDF + LogReg (English)"]})

## Model 2: Multilingual MiniLM embeddings + MLP (Kinyarwanda)

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.neural_network import MLPClassifier

encoder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
emb = encoder.encode(RW.tolist(), show_progress_bar=False, normalize_embeddings=True)
mlp = MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=500, random_state=RNG).fit(emb[tr], y[tr])
results["Model 2: MiniLM + MLP (Kinyarwanda)"] = score(y[te], mlp.predict(emb[te]))
print("Model 2:", results["Model 2: MiniLM + MLP (Kinyarwanda)"])
save_json("model2_metrics.json", {"Model 2: MiniLM + MLP (Kinyarwanda)": results["Model 2: MiniLM + MLP (Kinyarwanda)"]})

## Model 3: AfroXLMR fine-tune (Kinyarwanda)

In [ ]:
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

SAVE_MODEL = False  # set True to also export the fine-tuned model (large, ~1.1GB)
MODEL = "Davlan/afro-xlmr-base"
MAXLEN, BATCH, EPOCHS, LR = 128, 16, 4, 2e-5
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=len(labels)).to(DEVICE)

class IntentDataset(Dataset):
    def __init__(self, texts, ys):
        self.enc = tokenizer(list(texts), truncation=True, padding="max_length", max_length=MAXLEN, return_tensors="pt")
        self.y = torch.tensor(list(ys))
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.y[i]
        return item

counts = np.bincount(yi[tr], minlength=len(labels)); counts[counts == 0] = 1
weights = torch.tensor(counts.sum() / (len(labels) * counts), dtype=torch.float).to(DEVICE)
loss_fn = torch.nn.CrossEntropyLoss(weight=weights)

train_dl = DataLoader(IntentDataset(RW[tr], yi[tr]), batch_size=BATCH, shuffle=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * len(train_dl) * EPOCHS), len(train_dl) * EPOCHS)

for epoch in range(EPOCHS):
    model.train(); total = 0.0
    for batch in train_dl:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
        loss = loss_fn(out.logits, batch["labels"])
        loss.backward(); optimizer.step(); scheduler.step(); total += loss.item()
    print(f"epoch {epoch + 1}/{EPOCHS}  avg_loss={total / len(train_dl):.3f}")

model.eval(); preds = []
for batch in DataLoader(IntentDataset(RW[te], yi[te]), batch_size=32):
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    with torch.no_grad():
        preds += model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits.argmax(-1).cpu().tolist()
results["Model 3: AfroXLMR fine-tune (Kinyarwanda)"] = score(yi[te], preds)
print("\nModel 3:", results["Model 3: AfroXLMR fine-tune (Kinyarwanda)"], "\n")
print(classification_report(yi[te], preds, target_names=labels, zero_division=0))
save_json("kinyarwanda_finetune_metrics.json", {"Model 3: AfroXLMR fine-tune (Kinyarwanda)": results["Model 3: AfroXLMR fine-tune (Kinyarwanda)"]})

cm = confusion_matrix(yi[te], preds, labels=list(range(len(labels))))
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ConfusionMatrixDisplay(cm, display_labels=labels).plot(cmap="Greens", ax=ax, colorbar=False, xticks_rotation=45)
ax.set_title("Model 3 (AfroXLMR) confusion matrix")
plt.tight_layout(); save_fig("model3_confusion.png"); plt.show()

if SAVE_MODEL:
    model.save_pretrained("models/afroxlmr-intent"); tokenizer.save_pretrained("models/afroxlmr-intent")
    print("saved models/afroxlmr-intent")

## Model comparison

In [ ]:
comparison = pd.DataFrame({k: {"Accuracy": v["accuracy"], "F1": v["f1_macro"]} for k, v in results.items()}).T
print(comparison.to_string())
comparison.to_csv(os.path.join(REPORTS, "model_comparison.csv"))
print("saved", os.path.join(REPORTS, "model_comparison.csv"))
ax = comparison.plot.bar(figsize=(9, 4.5), rot=15, color=["#2A9D8F", "#E76F51"], edgecolor="white")
ax.set_ylim(0, 1.0); ax.set_ylabel("score"); ax.set_title("Model comparison")
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", fontsize=8, padding=2)
plt.tight_layout(); save_fig("model_comparison.png"); plt.show()

## Cross-lingual degradation

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate, StratifiedKFold

cv = StratifiedKFold(5, shuffle=True, random_state=RNG)
def cross_lingual_f1(stop_words, column):
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(stop_words=stop_words, ngram_range=(1, 2), min_df=2, max_features=8000, sublinear_tf=True)),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RNG)),
    ])
    r = cross_validate(pipe, df[column], y, cv=cv, scoring=["accuracy", "f1_macro"])
    return r["test_accuracy"].mean(), r["test_f1_macro"].mean()

en_acc, en_f1 = cross_lingual_f1("english", "Context")
rw_acc, rw_f1 = cross_lingual_f1(None, "context_rw")
print(f"English:     acc={en_acc:.3f}  F1={en_f1:.3f}")
print(f"Kinyarwanda: acc={rw_acc:.3f}  F1={rw_f1:.3f}")
print(f"Degradation: dF1={en_f1 - rw_f1:+.3f} ({(1 - rw_f1 / en_f1) * 100:.0f}% relative)")
save_json("cross_lingual_degradation.json", {
    "english": {"accuracy": round(float(en_acc), 4), "f1_macro": round(float(en_f1), 4)},
    "kinyarwanda_mt": {"accuracy": round(float(rw_acc), 4), "f1_macro": round(float(rw_f1), 4)},
    "delta_f1_macro": round(float(en_f1 - rw_f1), 4)})
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.bar(["English", "Kinyarwanda"], [en_f1, rw_f1], color=["#2A9D8F", "#E76F51"])
ax.set_ylabel("macro-F1"); ax.set_ylim(0, 0.8); ax.set_title("Cross-lingual degradation (same model)")
for i, value in enumerate([en_f1, rw_f1]):
    ax.text(i, value + 0.01, f"{value:.3f}", ha="center")
plt.tight_layout(); save_fig("cross_lingual_degradation.png"); plt.show()

## Confidence-gate calibration

In [ ]:
proba = clf.predict_proba(Xte)
pred = clf.classes_[proba.argmax(1)]
conf = proba.max(1)
rows = []
for t in [0.30, 0.35, 0.40, 0.45, 0.50]:
    answered = conf >= t
    acc = accuracy_score(y[te][answered], pred[answered]) if answered.any() else float("nan")
    rows.append({"threshold": t, "coverage": round(float(answered.mean()), 3), "acc_on_answered": round(float(acc), 3)})
sweep = pd.DataFrame(rows).set_index("threshold")
print(sweep.to_string())
sweep.to_csv(os.path.join(REPORTS, "threshold_sweep.csv"))
print("saved", os.path.join(REPORTS, "threshold_sweep.csv"))

## Save everything

In [ ]:
save_json("all_metrics.json", results)
import shutil
shutil.make_archive("umubyeyi_outputs", "zip", root_dir=".", base_dir="reports")
print("bundled -> umubyeyi_outputs.zip")
if IN_COLAB:
    from google.colab import files
    files.download("umubyeyi_outputs.zip")
    print("Downloaded. Extract it at your repo root so reports/ is populated.")
else:
    print("Saved under reports/ (and bundled into umubyeyi_outputs.zip).")